In [1]:
install.packages("table1")

Installing package into ‘/home/jupyter/.R/library’
(as ‘lib’ is unspecified)



In [2]:
library(bigrquery)
library(knitr)
library(tidyverse)
library(ggplot2)
library("table1")
library("IRdisplay")
bq_auth()

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.2     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.4     ✔ tibble    3.2.1
✔ lubridate 1.9.2     ✔ tidyr     1.3.0
✔ purrr     1.0.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘table1’


The following objects are masked from ‘package:base’:

    units, units<-




In [3]:
data <-bq_dataset_query(query="
SELECT *

 FROM `yhcr-prd-phm-bia-core.CB_1935_AK.src_bmbc_EYFSP_COMG02IMD`"
                       ,x="yhcr-prd-phm-bia-core.CB_1935_AK")

EYFSP2 <- bq_table_download(data) 

In [4]:
data1 <-bq_dataset_query(query="
select round(avg(count1),5)

from(

SELECT person_id,count(person_id) count1

 FROM `yhcr-prd-phm-bia-core.CB_1935_AK.src_bmbc_EYFSP_COMG01`

 group by person_id)base"
                       ,x="yhcr-prd-phm-bia-core.CB_1935_AK")

EYFSP2AVG <- bq_table_download(data1) 

In [5]:
EYFSP2$Gender[(EYFSP2$Gender) == "F"] <- "Female"
EYFSP2$Gender[(EYFSP2$Gender) == "M"] <- "Male"
EYFSP2$Gender[(EYFSP2$Gender) == "U"] <- "Unknown"

In [6]:
display_jupyter <- function(x) {
  css <- system.file("table1_defaults_1.0/table1_defaults.css", package="table1")
  css <- paste(readLines(css), collapse="\n")
  x <- htmltools::tagList(htmltools::tags$style(css), htmltools::tags$div(class="Rtable1", x))
  IRdisplay::display_html(as.character(x))
}

In [7]:
label(EYFSP2$AcademicYear)      <- "Academic Year"
label(EYFSP2$IMD)      <- "Average IMD"
label(EYFSP2$FSMEligible)      <- "FSM Eligiblity"
label(EYFSP2$ethnic_group)      <- "Ethnic Group"
label(EYFSP2$Gender)      <- "Gender"

In [8]:
 x<-table1(~ EYFSP2$AcademicYear + EYFSP2$FSMEligible+EYFSP2$Gender + EYFSP2$ethnic_group +EYFSP2$IMD|EYFSP2$score,data=EYFSP2
          ,justify=c("left", "left", rep("center", 5)),format_number = TRUE
          ,caption = "COSEY1: Development & Education -1.2 Speech,Language & Communication.Table presenting COMGO2 - Understanding results from the Early years foundation stage profile (EYFSP)  "
    
         )

In [9]:
display_jupyter(x)

,1 - Emerging(N=15000),2 - Expected level(N=45659),3 - Exceeded(N=14912),A - Exemption not assessed (N=184),No Data(N=23),Overall(N=75778)
Academic Year,,,,,,
2012/2013,2929 (19.5%),6758 (14.8%),1612 (10.8%),32 (17.4%),0 (0%),11331 (15.0%)
2013/2014,2766 (18.4%),6582 (14.4%),1994 (13.4%),18 (9.8%),0 (0%),11360 (15.0%)
2014/2015,2117 (14.1%),6762 (14.8%),2231 (15.0%),31 (16.8%),0 (0%),11141 (14.7%)
2015/2016,1906 (12.7%),6687 (14.6%),2313 (15.5%),0 (0%),23 (100%),10929 (14.4%)
2016/2017,1864 (12.4%),6531 (14.3%),2339 (15.7%),19 (10.3%),0 (0%),10753 (14.2%)
2017/2018,1805 (12.0%),6352 (13.9%),2157 (14.5%),52 (28.3%),0 (0%),10366 (13.7%)
2018/2019,1613 (10.8%),5987 (13.1%),2266 (15.2%),32 (17.4%),0 (0%),9898 (13.1%)
FSM Eligiblity,,,,,,
Yes,3922 (26.1%),8909 (19.5%),1728 (11.6%),38 (20.7%),6 (26.1%),14603 (19.3%)


In [10]:
OverallTrend1 <- EYFSP2 %>% select(1,5,7)

PersonalSocial1 <- OverallTrend1 %>%
  group_by(AcademicYear,score) 

PS1<- PersonalSocial1 %>% count(AcademicYear,score)


PS1<- group_by(PS1, AcademicYear) %>% mutate(percent = (n/sum(n))*100)


#PS1<-unique(filter(PS1,result == ))

#PS1<- group_by(PS1, AcademicYear) %>% mutate(percent = (n/sum(n))*100)

PS1<-unique(filter(PS1,score == "2 - Expected level" |score == "3 - Exceeded" ))
PS1 <- PS1 %>% select(1,4)

PS1<- group_by(PS1, AcademicYear) %>% mutate(PctPerYear =(sum(percent)))
PS1 <- PS1 %>% select(1,3)

PS1 <- distinct(PS1)



In [12]:
OVERALL<-ggplot(PS1 , aes(x = AcademicYear, y = PctPerYear,group = 1)) +
#scale_y_continuous( limits=c(0,1))+
  geom_line() +
  geom_point() +
   ggtitle(expression(atop("COS-EY 1: 1.2 Speech, Language & Communication",atop("COMGO2 - Understanding results from the Early years foundation stage profile (EYFSP)"), atop(italic("percentage of people achieveing atleast 'Expected' grade"), "")))) +


labs(
       x = "Academic Year",
       y = "Percentage of students per year achieving atleast the expected level") +
scale_y_continuous(labels = function(x) paste0(x, "%"),limits=c(0,100))+
               theme(panel.border = element_blank(),# panel.grid.major = element_blank(),
#panel.grid.minor = element_blank(), 
      axis.line = element_line(colour = "black"),axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1))+
geom_text(aes(label = round(PctPerYear,0)), hjust = 0.5,  vjust = -1) +
                   #+
  theme_minimal()

In [13]:
OverallTrend1 <- EYFSP2 %>% select(1,2,5,7)

PersonalSocial1 <- OverallTrend1 %>%
  group_by(AcademicYear,score,ethnic_group) 

PS1<- PersonalSocial1 %>% count(AcademicYear,score,ethnic_group)


PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(percent = (n/sum(n))*100)


#PS1<-unique(filter(PS1,result == ))

#PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(percent = (n/sum(n))*100)

PS1<-unique(filter(PS1,score == "2 - Expected level" |score == "3 - Exceeded" ))
PS1 <- PS1 %>% select(1,3,5)

PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(PctPerYear =round(sum(percent)),2)
PS1 <- PS1 %>% select(1,2,4)

PS1 <- distinct(PS1)


In [14]:
 ethnicity <- ggplot(PS1 , aes(x = AcademicYear, y = PctPerYear,group = ethnic_group,color = ethnic_group)) +
#scale_y_continuous( limits=c(0,1))+
  geom_line() +
  geom_point() +
   ggtitle(expression(atop("COS-EY 1: 1.2 Speech, Language & Communication", atop(italic("percentage of people achieveing atleast 'Expected' grade"),
                                                                                  atop(italic("By ethnic group")))))) +
  labs(
       x = "Academic Year",
       y = "Percentage of students per year achieving atleast the expected level") +
scale_y_continuous(labels = function(x) paste0(x, "%"),limits=c(0,100))+
               theme(panel.border = element_blank(),# panel.grid.major = element_blank(),
#panel.grid.minor = element_blank(), 
      axis.line = element_line(colour = "black"),axis.text.x = element_text(angle = 45, vjust = 1, hjust=1))+
geom_text(aes(label = round(PctPerYear,0)), hjust = 0.75,  vjust = -2) +
                   theme_minimal()+
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))

In [15]:
OverallTrend1 <- EYFSP2 %>% select(1,4,5,7)

PersonalSocial1 <- OverallTrend1 %>%
  group_by(AcademicYear,score,Gender) 

PS1<- PersonalSocial1 %>% count(AcademicYear,score,Gender)


PS1<- group_by(PS1, AcademicYear,Gender) %>% mutate(percent = (n/sum(n))*100)
#PS1<-unique(filter(PS1,result == ))

#PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(percent = (n/sum(n))*100)

PS1<-unique(filter(PS1,score == "2 - Expected level" |score == "3 - Exceeded" ))
PS1 <- PS1 %>% select(1,3,5)

PS1<- group_by(PS1, AcademicYear,Gender) %>% mutate(PctPerYear =round(sum(percent)),2)
PS1 <- PS1 %>% select(1,2,4)

PS1 <- distinct(PS1)
PS1 <-PS1<-unique(filter(PS1,Gender == "Female" |Gender == "Male" ))



In [16]:
gender <- ggplot(PS1 , aes(x = AcademicYear, y = PctPerYear,group = Gender,color = Gender)) +
#scale_y_continuous( limits=c(0,1))+
  geom_line() +
  geom_point() +
   ggtitle(expression(atop("COS-EY 1: 1.2 Speech, Language & Communication", atop(italic("percentage of people achieveing atleast 'Expected' grade"),
                                                                                  atop(italic("By Gender")))))) +
  labs(
       x = "Academic Year",
       y = "Percentage of students per year achieving atleast the expected level") +
scale_y_continuous(labels = function(x) paste0(x, "%"),limits=c(0,100))+
               theme(panel.border = element_blank(),# panel.grid.major = element_blank(),
#panel.grid.minor = element_blank(), 
      axis.line = element_line(colour = "black"),axis.text.x = element_text(angle = 45, vjust = 1, hjust=1))+
geom_text(aes(label = round(PctPerYear,0)), hjust = 0.75,  vjust = -1) +
                   theme_minimal()+
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))

In [17]:
OverallTrend1 <- EYFSP2 %>% select(1,10,5,7)

PersonalSocial1 <- OverallTrend1 %>%
  group_by(AcademicYear,score,IMD) 

PS1<- PersonalSocial1 %>% count(AcademicYear,score,IMD)


PS1<- group_by(PS1, AcademicYear,IMD) %>% mutate(percent = (n/sum(n))*100)
#PS1<-unique(filter(PS1,result == ))

#PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(percent = (n/sum(n))*100)

PS1<-unique(filter(PS1,score == "2 - Expected level" |score == "3 - Exceeded" ))
PS1 <- PS1 %>% select(1,3,5)

PS1<- group_by(PS1, AcademicYear,IMD) %>% mutate(PctPerYear =round(sum(percent)),2)
PS1 <- PS1 %>% select(1,2,4)

PS1 <- distinct(PS1)



In [18]:
 IMD <- ggplot(PS1 , aes(x = AcademicYear, y = PctPerYear,group = IMD,color = IMD)) +
#scale_y_continuous( limits=c(0,1))+
  geom_line() +
  geom_point() +
   ggtitle(expression(atop("COS-EY 1: 1.2 Speech, Language & Communication", atop(italic("percentage of people achieveing atleast 'Expected' grade"),
                                                                                  atop(italic("By IMD")))))) +
  labs(
       x = "Academic Year",
       y = "Percentage of students per year achieving atleast the expected level") +
scale_y_continuous(labels = function(x) paste0(x, "%"),limits=c(0,100))+
               theme(panel.border = element_blank(),# panel.grid.major = element_blank(),
#panel.grid.minor = element_blank(), 
      axis.line = element_line(colour = "black"),axis.text.x = element_text(angle = 45, vjust = 1, hjust=1))+
geom_text(aes(label = round(PctPerYear,0)), hjust = 0.75,  vjust = -1) +
                   theme_minimal()+
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))

In [19]:
library(grid)
#library(gridtext)
library(gridExtra)
plot <- list(OVERALL, gender, IMD, ethnicity)
pdf('1.2 Speech Language and Communication2.pdf',width=10, height=10)
  title <- "Plots for 1.2 Speech Language & Communication : COMGO2 Communication and Language - Understanding"
grid.text(title) 
plot
dev.off()
#


Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine




[[1]]

[[2]]

[[3]]

[[4]]


png 
  2